# Plitt & Giocomo (2021) on dandiset 000054, part 1: sessions, behaviour and place cells

Paper. Plitt MH, Giocomo LM (2021). Experience-dependent contextual codes in the hippocampus.
*Nature Neuroscience* 24, 705-714.
[doi:10.1038/s41593-021-00816-6](https://doi.org/10.1038/s41593-021-00816-6). The claims file was
written from the NIH author manuscript in PubMed Central (PMC8893323), with the Methods, the main
and Extended Data legends and the supplementary information; it is the accepted manuscript, not
the version of record.

Dandiset. [DANDI:000054](https://dandiarchive.org/dandiset/000054/0.260415.1712), version
0.260415.1712. The paper's data statement points to the draft and cites no version. Of the six
published versions, the three of 2021-08-19 hold 85 files (1.96 TB, raw imaging included) for 10
mice only (F1, F2, F3, F5 and R1 to R6). The three of April 2026 hold processed files for all 16
mice. The most recent, 0.260415.1712 (126 files, 112 GB), is pinned: the protocol asks for the
most recent published version, and no earlier one matches the claims file better, as the 2021
versions lack six of the twelve rare and frequent morph mice and all four decision mice. It
differs from 0.260409.0433 in mouse R5 only (training sessions 1, 5 and 7 removed and the other
R5 files replaced); the cell below lists the difference.

What the files hold. One file per imaging session: the deconvolved activity of every Suite2P
cell, the VR behaviour aligned to the imaging frames (imaged trials only), and a JSON annotation
with per-trial values. The non-imaged trials, and the sessions without imaging, are not in the
dandiset.

Outside input. The authors' "Unprocessed VR data" on figshare
([doi:10.6084/m9.figshare.31974756.v1](https://doi.org/10.6084/m9.figshare.31974756.v1), one zip of
1.09 GB, published 2026-04-09) holds every VR session, imaged or not, with a session database
(header M7 of the claims file, answer of 2026-10-22). It is read here by HTTP range requests, file
by file, into memory, in part 2 (`02_population_inference.ipynb`), which tests its link to the
dandiset (vr-deposit-link) and training-intermediate-fraction, which uses the deposit alone
(`data: elsewhere`).

This notebook tests the claims on sessions, behaviour and place cells. The population, SF and
inference claims are in part 2 (`02_population_inference.ipynb`), which also holds the summary.

| Claim | Notebook | Status |
|---|---|---|
| cells-per-session | 1 | tested |
| session-and-trial-counts | 1 | tested |
| vr-deposit-link | 2 | tested (link claim, figshare deposit) |
| training-intermediate-fraction | 2 | tested (figshare deposit, data elsewhere) |
| place-cell-fractions | 1 | tested |
| edf4a-fewer-place-cells-frequent | 1 | tested |
| edf4a-overlap-above-chance | 1 | tested |
| edf4bc-overlap-cells-at-ends-and-reward | 1 | tested |
| fig2eh-place-cell-similarity-differs | 1 | tested |
| edf5-umap-rare-frequent-separable | 1 | tested |
| edf1-licking-before-reward | 1 | tested |
| edf6-nmf-confirms-remapping | 1 | tested |
| fig3ab, edf7, fig3f, fig4 | 2 | tested |
| fig6a, fig6bc, fig6d, map-estimate-values, edf9, edf9k, edf9l, edf10, fig7g, edf3 | 2 | tested (priors from the deposit, data with inputs) |
| fig7-model-reproduces-remapping | 2 | tested (priors from the deposit, data elsewhere) |
| edf2-fh-best-predictor | - | blocked: data missing (renders of the VR scenes) |
| supp-earlier-morph-studies | - | blocked: other (uses no data of this paper) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import morph as M

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
started = time.time()

assets = M.asset_table()
print(f"dandiset {M.DANDISET_ID} version {M.DANDISET_VERSION}: {len(assets)} NWB files, "
      f"{assets['size'].sum() / 1e9:.1f} GB")
testing_rf = assets[(assets.group != "decision") & (assets.kind == "testing")]
others = assets.drop(testing_rf.index)
place = M.process_many([r for _, r in testing_rf.iterrows()], "place")
print(f"rare and frequent sessions 8 to N, streamed with place cells: {len(place)} in "
      f"{time.time() - started:.0f} s")
meta = M.process_many([r for _, r in others.iterrows()], "meta", workers=8)
print(f"other files, behaviour and trial annotation only: {len(meta)}; total {time.time() - started:.0f} s")
sessions = sorted(place + meta, key=lambda s: (s["group"], s["mouse"], s["day"], s["part"]))
RARE = ["R1", "R2", "R3", "R4", "R5", "R6"]
FREQ = ["F1", "F2", "F3", "F4", "F5", "F6"]

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
from dandi.dandiapi import DandiAPIClient

with DandiAPIClient() as client:
    prev = {a.path: a.size for a in client.get_dandiset(M.DANDISET_ID, "0.260409.0433").get_assets()}
cur = dict(zip(assets.path, assets["size"]))
print("Difference from version 0.260409.0433 (path: earlier size -> pinned size):")
for p in sorted(set(prev) | set(cur)):
    if prev.get(p) != cur.get(p):
        print(f"  {p}: {prev.get(p)} -> {cur.get(p)}")

info = pd.DataFrame([{
    "mouse": s["mouse"], "group": s["group"], "kind": s["kind"], "day": s["day"], "part": s["part"],
    "json_day": s["json_day"], "session_id": s["session_id"], "frames": s["n_frames"], "cells": s["n_cells"],
    "trials": len(s["trials"]), "sex": s["subject"]["sex"], "genotype": s["subject"]["genotype"],
    "description": s["subject"]["description"], "rate": s["rate"], "imaging_rate": s["imaging_rate"],
    "dt_s": s["dt"], "start_time": s["start_time"][:10], "same_ts": s["same_ts"],
    "starts_match": s["starts_match"], "stops_match": s["stops_match"],
    "trial_numbers_match": s["trial_numbers_match"], "morph_series_match": s["morph_series_match"],
    "fingerprint": s["fingerprint"]} for s in sessions])
print("\nFiles by group and session kind:")
print(pd.crosstab(info.group, info.kind).to_string())
print("\nSessions (day as in the file name; part for the two files of session 1 of R1 to R3):")
print(info.groupby(["group", "mouse"]).apply(lambda d: " ".join(
    f"{k[0]}{dd}" + (f".{pp}" if pp > 1 else "") for k, dd, pp in zip(d.kind, d.day, d.part))).to_string())
subj = info.drop_duplicates("mouse")[["mouse", "group", "sex", "genotype", "description"]]
print("\nSubjects:")
print(subj.to_string(index=False))
print("Sex counts:", subj.sex.value_counts().to_dict(), "(paper: 16 mice, 9 male and 7 female)")
print("\nFile name day equals the JSON 'day' (or day.part):",
      bool(np.allclose(info.json_day, info.day + (info.part - 1) / 10 + np.where(info.mouse.isin(["R1", "R2", "R3"]) & (info.day == 1), 0.1, 0)))
      or sorted(set(zip(info.day, info.part, info.json_day)))[:6])
print("Imaging rate (Hz), all files:", sorted(set(info.rate.round(4))), "; frame interval (s):",
      sorted(set(info.dt_s.round(5))), "(paper: 15.46 Hz)")
print("Every behaviour series shares one timestamp array:", bool(info.same_ts.all()))
print("Trial starts and teleports of the JSON match the flag series:", bool(info.starts_match.all()),
      bool(info.stops_match.all()), "; trial number series constant within each trial:",
      bool(info.trial_numbers_match.all()), "; morph series equals the JSON morph:", bool(info.morph_series_match.all()))
print("session_start_time dates:", sorted(set(info.start_time)))
dup = info[info.duplicated(["frames", "cells", "fingerprint"], keep=False)]
print("Files sharing frame count, cell count and the sum of activity in frames 1000 to 1009:",
      len(dup), "(none means no recording is deposited twice)")
print(f"Cells per file: {info.cells.min()} to {info.cells.max()}; trials per file: {info.trials.min()} to {info.trials.max()}")

In [ ]:
rng_rows = []
for s in place:
    r = s["ranges"]
    rng_rows.append({"mouse": s["mouse"], "day": s["day"], "pos_min": r["position"][0], "pos_max": r["position"][1],
                     "speed_min": r["speed"][0], "speed_max": r["speed"][1], "licks_max": r["licks"][1],
                     "act_min": s["act_min"], "act_max": s["act_max"], "norm_check": s["norm_check"],
                     "p99_zero": s["p99_zero"], "speed_kept": s["speed_kept"], "act_check": s["act_check"]})
rr = pd.DataFrame(rng_rows)
print("Units:", place[0]["units"])
print(rr.describe().loc[["min", "max"]].round(3).to_string())
tr_all = pd.concat([s["trials"].assign(mouse=s["mouse"], group=s["group"], day=s["day"], kind=s["kind"]) for s in sessions])
print("\nS-tilde values by group and kind:")
print(tr_all.groupby(["group", "kind"]).morph.apply(lambda m: sorted(set(np.round(m, 3)))[:8]
                                                   + (["..."] if m.nunique() > 8 else [])).to_string())
for col in ["wall", "tower", "background"]:
    print(f"{col} jitter range: {tr_all[col].min():.3f} to {tr_all[col].max():.3f} (Methods: -0.1 to 0.1)")
print(f"S = S-tilde + wall jitter: {tr_all.S.min():.3f} to {tr_all.S.max():.3f}; f_h {tr_all.fh.min():.2f} to "
      f"{tr_all.fh.max():.2f}; units {tr_all.unit.min():.3f} to {tr_all.unit.max():.3f}")
print("effective_morph of the JSON equals S:", bool(np.allclose(tr_all.effective_morph, tr_all.S)))
rw = tr_all[tr_all.kind == "testing"].reward_pos
print(f"Reward position, testing sessions: {rw.min():.1f} to {rw.max():.1f} cm, missing on {rw.isna().mean():.1%} "
      f"of trials; omission flag on {tr_all[tr_all.kind == 'testing'].omission.mean():.1%}")

Behaviour: position runs from 0 to 450 cm in a trial, with -50 cm in the inter-trial tunnel (and
a few frames at -500 cm, outside any trial), as described. Speed is in cm/s and is negative on
some frames. Deconvolved activity is non-negative. The normalised series matches the
description (each cell divided by its 99th percentile; `norm_check` is the largest difference
over a block of 100 frames). The jitters lie in -0.1 to 0.1 and S in -0.1 to 1.1, as in the
Methods, and the JSON `effective_morph` is S. The decision mice FD1's training sessions 1 and 3
hold continuous morph values, not the five S-tilde values.

Metadata against the data and the paper: the imaging rate is 15.4609 Hz, as the paper gives. The
`session_start_time` of every file is the date of conversion in 2026, not of the recording, so
calendar days between sessions cannot be read from the files. The files hold imaging sessions
5 and 7 (and R4's session 6), which the paper does not list among imaging sessions (1, 3 and 8
to N). R6 has no training files and R4 and R5 lack session 1, and FD3 has no testing sessions.
Rare morph mice see intermediate values only in sessions 3 and 8 to N, as described.

In [ ]:
lag = pd.DataFrame([{"mouse": s["mouse"], "day": s["day"], **{f"lag {k}": v for k, v in s["lag_rel"].items()}} for s in place])
best = lag.filter(like="lag").idxmax(axis=1).str.replace("lag ", "").astype(int)
print("Clock check 1: median over the first 300 cells of the odd-even trial map correlation, with activity shifted "
      "by a lag (frames of 65 ms; positive = activity later) against position. Mean over sessions:")
print(lag.filter(like="lag").mean().round(3).to_string())
print("Lag of the maximum, per session:", best.value_counts().sort_index().to_dict())
ls = pd.DataFrame([{"mouse": s["mouse"], "day": s["day"], **s["label_shift"]} for s in place])
print("\nClock check 2: mean SF of S-tilde = 1 minus S-tilde = 0 trials, with the true trial labels and with each "
      "trial given the next trial's label:")
print(ls[["true", "shifted by one trial"]].describe().loc[["min", "50%", "max"]].round(3).to_string())
print("sessions where the true labels give the larger contrast:", int((ls["true"] > ls["shifted by one trial"]).sum()), "of", len(ls))

The split-half map correlation is highest within 8 frames (0.5 s) of zero lag in most sessions and
falls at lags of 1 to 4 s, and in every session the S-tilde contrast in the activity is present
with the true trial labels and gone when the labels are moved by one trial. So the activity, the
position and the trial table are consistent with one shared clock in every session, to within
about half a second; the map correlation cannot resolve a smaller offset, as the mice run at a
similar speed on every trial.

## cells-per-session

> Each imaging session recorded between 161 and 3,657 putative CA1 pyramidal cells.

In [ ]:
info["paper_imaging"] = (info.day == 1) | (info.day == 3) | (info.day >= 8)
rows = []
for reading, groups in (("a: R and F", ["rare", "frequent"]), ("b: R, F and FD", ["rare", "frequent", "decision"])):
    for sel_name, sel in (("sessions 1, 3 and 8 to N", info.paper_imaging), ("every file", info.paper_imaging | True)):
        x = info[info.group.isin(groups) & sel]
        rows.append({"reading": reading, "files": sel_name, "n": len(x), "min": x.cells.min(), "max": x.cells.max(),
                     "min_file": f"{x.loc[x.cells.idxmin(), 'mouse']} day {x.loc[x.cells.idxmin(), 'day']}",
                     "max_file": f"{x.loc[x.cells.idxmax(), 'mouse']} day {x.loc[x.cells.idxmax(), 'day']}",
                     "min within 161 +- 16.1": abs(x.cells.min() - 161) <= 16.1,
                     "max within 3657 +- 365.7": abs(x.cells.max() - 3657) <= 365.7})
print(pd.DataFrame(rows).to_string(index=False))
print("\nCells per session, by mouse (all files):")
print(info.groupby("mouse").cells.agg(["min", "max", "size"]).to_string())

**Result: reproduced.** Over the sessions the paper describes as imaged (1, 3 and 8 to N), the
rare and frequent morph mice have 161 to 3,653 cells per session (F3 session 8 and F6 session 3),
within tolerance of 161 and 3,657; with the decision mice added the range is the same. The
dandiset also holds imaging of sessions 5 and 7, which the paper does not describe; with them the
maximum is 4,103 cells (F6 session 5), outside the tolerance.

## session-and-trial-counts

> The analyses of sessions 8 to N use 24 rare morph and 21 frequent morph sessions, holding 2,610
> and 1,986 trials; the four decision mice contribute 2,460 trials.

In [ ]:
late = info[info.day >= 8]
for g in ["rare", "frequent"]:
    x = late[late.group == g]
    print(f"{g}: {len(x)} sessions 8 to N, {x.trials.sum()} trials; per mouse:",
          x.groupby("mouse").agg(sessions=("day", "size"), trials=("trials", "sum")).to_dict("index"))
fd = info[info.group == "decision"]
print(f"decision mice, reading (a) every imaged session: {len(fd)} sessions, {fd.trials.sum()} trials; "
      f"(b) sessions 8 to N: {(fd.day >= 8).sum()} sessions, {fd[fd.day >= 8].trials.sum()} trials")
print("decision mice per mouse:", fd.groupby("mouse").agg(sessions=("day", "size"), trials=("trials", "sum"),
                                                          late=("day", lambda d: int((d >= 8).sum()))).to_dict("index"))
exp = {"rare sessions": (24, 2.4), "frequent sessions": (21, 2.1), "rare trials": (2610, 261),
       "frequent trials": (1986, 198.6), "decision trials": (2460, 246)}
for reading, fdx in (("a", fd), ("b", fd[fd.day >= 8])):
    got = {"rare sessions": (late.group == "rare").sum(), "frequent sessions": (late.group == "frequent").sum(),
           "rare trials": late[late.group == "rare"].trials.sum(), "frequent trials": late[late.group == "frequent"].trials.sum(),
           "decision trials": fdx.trials.sum()}
    print(f"reading {reading}:", {k: (int(v), bool(abs(v - exp[k][0]) <= exp[k][1])) for k, v in got.items()})

**Result: partially reproduced.** Sessions 8 to N number 26 for the rare morph mice (24
expected, tolerance 2.4) and 21 for the frequent morph mice (21), holding 2,800 trials (2,610 +-
261) and 1,986 trials (1,986): these four counts are within tolerance. The decision mice
contribute 3,217 trials over every imaged session (reading a) and 2,022 over sessions 8 to N
(reading b), both outside 2,460 +- 246. FD3 has no session after 7 in the dandiset, and FD1's
sessions 1 and 3 hold continuous morph values.

## place-cell-fractions

> Place cells tiled the track in both conditions; 15,829 rare morph place cells (43% of recorded
> cells; 23% S-tilde = 0 and 21% S-tilde = 1 place cells) and 8,933 frequent morph place cells
> (30%; 11% and 10%).

Place cells per session and S-tilde, from the spatial information against 1,000 within-trial
circular shuffles (helper `place_cells`). Reading (a) leaves out frames below 2 cm/s, reading
(b) keeps all frames. The place cells the authors' 2026 code stored in each file's JSON
annotation are shown for comparison; they do not decide.

In [ ]:
pc_rows = []
for s in place:
    for g in "ab":
        m = s["pc"][g]["mask"]
        pc_rows.append({"mouse": s["mouse"], "group": s["group"], "day": s["day"], "reading": g, "cells": s["n_cells"],
                        "any": int(m.any(0).sum()), "s0": int(m[0].sum()), "s1": int(m[4].sum()),
                        "both": int((m[0] & m[4]).sum()), **{f"m{k}": int(m[k].sum()) for k in range(5)}})
pcd = pd.DataFrame(pc_rows)
expected = {"rare": (15829, 0.43, 0.23, 0.21), "frequent": (8933, 0.30, 0.11, 0.10)}
tol = {"rare": (1582.9, 0.043, 0.023, 0.021), "frequent": (893.3, 0.03, 0.011, 0.01)}
summary = []
for g in "ab":
    for den in ("pooled", "session mean"):
        row = {"speed reading": g, "denominator": den}
        allok, countok, fracok = True, True, True
        for grp in ("rare", "frequent"):
            x = pcd[(pcd.reading == g) & (pcd.group == grp)]
            if den == "pooled":
                vals = (x["any"].sum(), x["any"].sum() / x.cells.sum(), x.s0.sum() / x.cells.sum(), x.s1.sum() / x.cells.sum())
            else:
                vals = (x["any"].sum(), (x["any"] / x.cells).mean(), (x.s0 / x.cells).mean(), (x.s1 / x.cells).mean())
            ok = [abs(v - e) <= t for v, e, t in zip(vals, expected[grp], tol[grp])]
            row.update({f"{grp} count": int(vals[0]), f"{grp} any": round(vals[1], 3), f"{grp} S0": round(vals[2], 3),
                        f"{grp} S1": round(vals[3], 3)})
            countok &= ok[0]
            fracok &= all(ok[1:])
        row.update({"counts ok": countok, "fractions ok": fracok})
        summary.append(row)
print(pd.DataFrame(summary).to_string(index=False))
print("\nRecorded cells, sessions 8 to N:", pcd[pcd.reading == "a"].groupby("group").cells.sum().to_dict(),
      "(the paper's counts and fractions imply about 36,800 rare and 29,800 frequent)")
print("\nFraction of cells that are place cells for any S-tilde, per mouse (mean over sessions):")
print(pcd.assign(f=pcd["any"] / pcd.cells).pivot_table(index="mouse", columns="reading", values="f").round(3).to_string())
print("\nPlace cells per S-tilde, pooled fraction, reading a:")
print(pcd[pcd.reading == "a"].groupby("group")[[f"m{k}" for k in range(5)]].sum().div(
    pcd[pcd.reading == "a"].groupby("group").cells.sum(), axis=0).round(3).to_string())
print("\nShuffles run per cell (a cell stops once more than 50 shuffles reach its SI for every S-tilde and reading):",
      int(np.median(np.concatenate([s["pc"]["shuffles_run"] for s in place]))), "median")

In [ ]:
# Place-cell masks of the authors' 2026 code, as stored in the JSON annotation of each file (a comparison).
import h5py, json, remfile  # noqa: E401
jrows = []
def json_masks(url):
    with h5py.File(remfile.File(url), "r") as f:
        pci = json.loads(f["acquisition/trial_data/data"][0])["place_cell_info"]["masks"]
    return np.array([pci[k] for k in ["0.0", "0.25", "0.5", "0.75", "1.0"]], bool)
for s, (_, row) in zip(place, testing_rf.iterrows()):
    jm = json_masks(row.url)
    agree = {g: float((s["pc"][g]["mask"] == jm).mean()) for g in "ab"}
    jrows.append({"group": s["group"], "cells": s["n_cells"], "any": int(jm.any(0).sum()), "s0": int(jm[0].sum()),
                  "s1": int(jm[4].sum()), "agree_a": agree["a"], "agree_b": agree["b"]})
jd = pd.DataFrame(jrows)
for grp, x in jd.groupby("group"):
    print(f"{grp}: 2026-code place cells {x['any'].sum()} ({x['any'].sum() / x.cells.sum():.3f} of cells; S0 "
          f"{x.s0.sum() / x.cells.sum():.3f}, S1 {x.s1.sum() / x.cells.sum():.3f}); agreement of masks with reading a "
          f"{x.agree_a.mean():.3f}, reading b {x.agree_b.mean():.3f}")

**Result: not reproduced.** The recorded cells match the paper's implied totals (36,550 rare and
30,018 frequent against about 36,800 and 29,800), but far more of them are place cells. With
low-speed frames left out (reading a), 25,053 rare morph cells are place cells for some S-tilde
(68.5% pooled; 39.2% for S-tilde = 0 and 36.0% for S-tilde = 1) and 17,025 frequent morph cells
(56.7%; 24.0% and 24.1%). With all frames (reading b) the counts are 21,945 (60.0%; 31.0% and
28.2%) and 14,345 (47.8%; 17.8% and 16.3%). No combination of readings brings any of the eight
values within tolerance; per-session means give the same. The place cells the authors' 2026 code
stored in the files give similar fractions (rare 61.5%, frequent 48.6%) and agree with reading
(b) for about 96% of cell and S-tilde pairs, so the paper's lower fractions came from a
procedure that differs from both. That place cells tile the track is not tested by the criterion.

## edf4a-fewer-place-cells-frequent

> A smaller proportion of recorded cells were place cells in the frequent morph than in the rare
> morph condition.

In [ ]:
rows = []
for g in "ab":
    for den in ("pooled", "session mean"):
        vals = {}
        for grp in ("rare", "frequent"):
            x = pcd[(pcd.reading == g) & (pcd.group == grp)]
            if den == "pooled":
                vals[grp] = [x[c].sum() / x.cells.sum() for c in ("any", "s0", "s1")]
            else:
                vals[grp] = [(x[c] / x.cells).mean() for c in ("any", "s0", "s1")]
        rows.append({"speed": g, "denominator": den, **{f"rare {c}": round(v, 3) for c, v in zip(("any", "S0", "S1"), vals["rare"])},
                     **{f"freq {c}": round(v, 3) for c, v in zip(("any", "S0", "S1"), vals["frequent"])},
                     "frequent below rare (all three)": all(f < r for f, r in zip(vals["frequent"], vals["rare"]))})
print(pd.DataFrame(rows).to_string(index=False))
pm = pcd.assign(f=pcd["any"] / pcd.cells).groupby(["reading", "group", "mouse"]).f.mean().round(3)
print("\nPer-mouse fraction (any S-tilde):")
print(pm.unstack("reading").to_string())
for g in "ab":
    r, f = pm.loc[(g, "rare")], pm.loc[(g, "frequent")]
    print(f"reading {g}: rare mice {r.min():.3f} to {r.max():.3f}, frequent {f.min():.3f} to {f.max():.3f}; "
          f"Mann-Whitney p = {stats.mannwhitneyu(r, f).pvalue:.3g} (not the paper's test; it reports none)")

**Result: reproduced.** Under every reading the frequent morph fraction is below the rare morph
fraction, for any S-tilde and for S-tilde = 0 and = 1 (pooled, reading a: 0.567 against 0.685;
0.240 against 0.392; 0.241 against 0.360). Per mouse the ranges overlap (reading a: rare 0.62 to
0.79, frequent 0.41 to 0.71).

## edf4a-overlap-above-chance

> The fraction of cells that are place cells in both S-tilde = 0 and S-tilde = 1 trials (6.07%
> rare, 1.98% frequent) slightly exceeds the overlap expected of independent populations (4.78%
> and 1.06%), so the two extreme maps are largely independent (global remapping).

In [ ]:
exp4 = {"rare actual": (0.0607, 0.00607), "frequent actual": (0.0198, 0.00198),
        "rare chance": (0.0478, 0.00478), "frequent chance": (0.0106, 0.00106)}
rows = []
for g in "ab":
    x = pcd[pcd.reading == g].copy()
    x["actual"] = x.both / x.cells
    x["chance"] = (x.s0 / x.cells) * (x.s1 / x.cells)
    t = stats.ttest_1samp(x.chance - x.actual, 0)
    for den in ("pooled", "session mean"):
        v = {}
        for grp in ("rare", "frequent"):
            y = x[x.group == grp]
            if den == "pooled":
                v[f"{grp} actual"] = y.both.sum() / y.cells.sum()
                v[f"{grp} chance"] = (y.chance * y.cells).sum() / y.cells.sum()
            else:
                v[f"{grp} actual"] = y.actual.mean()
                v[f"{grp} chance"] = y.chance.mean()
        rows.append({"speed": g, "denominator": den, **{k: round(val, 4) for k, val in v.items()},
                     "t": round(t.statistic, 2), "p": f"{t.pvalue:.2g}", "sessions": len(x),
                     "actual above chance": bool((x.actual > x.chance).mean() > 0.5) and t.statistic < 0,
                     "four within tolerance": all(abs(v[k] - exp4[k][0]) <= exp4[k][1] for k in exp4)})
    print(f"reading {g}: sessions with actual above chance {int((x.actual > x.chance).sum())} of {len(x)}; "
          f"median actual/chance {np.median(x.actual / x.chance):.2f}")
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** The test holds: the overlap of S-tilde = 0 and = 1 place cells
exceeds the product of the two fractions (one-sample t-test of chance minus actual over 47
sessions, t = -10.08, p = 3.2e-13 with low-speed frames left out; t = -7.67, p = 9.2e-10 with all
frames; actual above chance in 44 of 47 sessions). The four values are outside tolerance under
every reading, as the place-cell fractions are higher than the paper's (pooled, reading b: rare
10.3% against chance 8.9%, frequent 4.2% against 3.4%; the paper 6.07%, 4.78%, 1.98% and 1.06%).
The median session's overlap is 1.15 (reading a) to 1.19 (reading b) times chance, so the two
maps are largely independent, as stated.

## edf4bc-overlap-cells-at-ends-and-reward

> Cells with place fields in both extreme environments (2,219 rare, 594 frequent) tended to
> cluster at the end points of the track and in the reward zone.

Answer of 2026-10-22: the paper gives no measure, so the criterion names two readings of how a
cell's two peaks are counted, (i) each environment's peaks separately and (ii) cells with both
peaks in the regions against the product of the two references, and reference (b) is now taken
per environment. Every combination of end points, peak reading and reference has to hold, in both
conditions; both speed readings of M3 are run, as for every place-cell claim.

In [ ]:
reward_bins = np.arange(25, 40)            # 250 to 400 cm
ends = {"a: first and last bin": np.array([0, 44]), "b: first and last 50 cm": np.r_[0:5, 40:45]}


def peak_bins(s, g, k):
    return np.nanargmax(np.nan_to_num(s["pc"][g]["maps"][k], nan=-1), axis=0)


rows = []
for g in "ab":
    for grp in ("rare", "frequent"):
        p0, p1, ref0, ref1 = [], [], [], []
        for s in place:
            if s["group"] != grp:
                continue
            m = s["pc"][g]["mask"]
            both = m[0] & m[4]
            pk0, pk1 = peak_bins(s, g, 0), peak_bins(s, g, 4)
            p0.append(pk0[both]); p1.append(pk1[both]); ref0.append(pk0[m[0]]); ref1.append(pk1[m[4]])
        p0, p1, ref0, ref1 = map(np.concatenate, (p0, p1, ref0, ref1))
        for ename, eb in ends.items():
            region = np.union1d(eb, reward_bins)
            in0, in1 = np.isin(p0, region), np.isin(p1, region)
            track = len(region) / 45
            r0, r1 = np.isin(ref0, region).mean(), np.isin(ref1, region).mean()
            tests = {("i: S0 peaks", "a"): (in0.mean(), track), ("i: S0 peaks", "b"): (in0.mean(), r0),
                     ("i: S1 peaks", "a"): (in1.mean(), track), ("i: S1 peaks", "b"): (in1.mean(), r1),
                     ("ii: both peaks", "a"): ((in0 & in1).mean(), track ** 2),
                     ("ii: both peaks", "b"): ((in0 & in1).mean(), r0 * r1)}
            for (peaks, ref), (share, refv) in tests.items():
                rows.append({"speed": g, "group": grp, "overlap cells": len(p0), "end points": ename[:1],
                             "peaks": peaks, "reference": ref, "share": round(share, 3), "ref value": round(refv, 3),
                             "ratio": round(share / refv, 3), "exceeds": bool(share > refv)})
r4 = pd.DataFrame(rows)
print(r4.to_string(index=False))
print("\nCombinations that exceed their reference, by speed reading and condition (of 12 each):")
print(r4.groupby(["speed", "group"]).exceeds.agg(["sum", "size"]).to_string())
print("Every combination holds in both conditions: reading a", bool(r4[r4.speed == "a"].exceeds.all()),
      "; reading b", bool(r4[r4.speed == "b"].exceeds.all()))
print("Failing combinations:")
print(r4[~r4.exceeds][["speed", "group", "end points", "peaks", "reference", "share", "ref value"]].to_string(index=False))
for g in "ab":
    x = r4[r4.speed == g].drop_duplicates("group")
    print(f"reading {g}: overlap cells rare {int(x[x.group == 'rare']['overlap cells'].iloc[0])} (2219 +- 221.9), "
          f"frequent {int(x[x.group == 'frequent']['overlap cells'].iloc[0])} (594 +- 59.4)")
# where the peaks fall, reading b, as 50 cm histograms
for grp in ("rare", "frequent"):
    for k, lab in ((0, "S0"), (4, "S1")):
        pk = np.concatenate([peak_bins(s, "b", k)[(s["pc"]["b"]["mask"][0] & s["pc"]["b"]["mask"][4])]
                             for s in place if s["group"] == grp])
        print(grp, lab, "peak positions of overlap cells, counts per 50 cm, reading b:",
              np.histogram(pk * 10 + 5, bins=np.arange(0, 451, 50))[0].tolist())

**Result: not reproduced.** Under the readings the answer added, the overlap cells' peaks fall in
the end points or the reward zone more often than the track share of those regions in every
combination (ratios 1.41 to 1.87), and cells with both peaks there exceed the product of the
references in every combination (ratios 1.02 to 3.48). Against the place cells of the same
environment (reference b), with the end points as the first and last bin, the separate peaks fall
short in 4 of the 48 combinations: rare S-tilde = 0 peaks 0.577 against 0.585 (low-speed frames
left out) and 0.618 against 0.627 (all frames), and frequent S-tilde = 0 and = 1 peaks 0.619
against 0.635 and 0.574 against 0.601 (low-speed frames left out). The criterion asks every
combination to hold in both conditions, so it is not met; where the overlap cells exceed the
place cells of their environment, it is by at most 10% (ratios 1.004 to 1.098). The counts of
overlap cells are far above the paper's (rare 5,997 and frequent 2,552 with low-speed frames left
out, 3,759 and 1,272 with all frames, against 2,219 and 594), as the place cells are
(place-cell-fractions).

## fig2eh-place-cell-similarity-differs

> Rare morph place cells keep their map up to a threshold morph value (S about 0.25 to 0.5) and
> then switch coherently, while frequent morph place cells change gradually; the difference shows
> in the place-cell similarity matrices across S-tilde values.

In [ ]:
off = ~np.eye(5, dtype=bool)


def cell_matrices(s, g):
    m = s["pc"][g]["mask"].any(0)
    maps = np.transpose(M.nan_smooth(s["pc"][g]["maps"][:, :, m], M.SMOOTH_SIGMA), (2, 0, 1))   # cells x 5 x bins
    a = M.unit_rows(maps)
    C = np.einsum("nij,nkj->nik", a, a)
    vals = C[:, off]
    z = (C - vals.mean(1)[:, None, None]) / vals.std(1)[:, None, None]
    return C, z


def session_joint_z(C):
    vals = C[:, off]
    return (C - vals.mean()) / vals.std()


results = {}
for g in "ab":
    sums = {"raw": {}, "z": {}, "zsession": {}}
    counts = {}
    for s in place:
        C, z = cell_matrices(s, g)
        for k, v in (("raw", C), ("z", z), ("zsession", session_joint_z(C))):
            sums[k][s["mouse"]] = sums[k].get(s["mouse"], 0) + np.nan_to_num(v).sum(0)
        counts[s["mouse"]] = counts.get(s["mouse"], 0) + len(C)
    results[g] = (sums, counts)


def pooled_perm(sums, counts, rare, freq):
    mice = rare + freq
    S = np.array([sums[m] for m in mice]); n = np.array([counts[m] for m in mice], float)
    def diff(mask):
        return S[mask].sum(0) / n[mask].sum() - S[~mask].sum(0) / n[~mask].sum()
    obs = diff(np.isin(mice, rare))
    import itertools
    perms = []
    for combo in itertools.combinations(range(len(mice)), len(rare)):
        mask = np.zeros(len(mice), bool); mask[list(combo)] = True
        if set(np.array(mice)[mask]) == set(rare):
            continue
        perms.append(diff(mask))
    return obs, np.array(perms)


for g in "ab":
    sums, counts = results[g]
    print(f"=== speed reading {g}; place cells per mouse:", counts)
    for kind, rare, freq, label in (("z", RARE, FREQ, "z-scored per cell (Methods)"),
                                    ("raw", RARE, FREQ, "raw"),
                                    ("zsession", RARE, FREQ, "variant: z-scored jointly per session (2020 code)"),
                                    ("z", RARE[:5], FREQ + ["R6"], "variant: R6 counted as frequent (2020 code)")):
        obs, perms = pooled_perm(sums[kind], counts, rare, freq)
        o_rng = M.outside_range(obs, perms) & off
        o_95 = M.outside_95(obs, perms) & off
        print(f"{label}: rare minus frequent")
        print(pd.DataFrame(obs, index=M.MORPHS, columns=M.MORPHS).round(3).to_string())
        print("  entries outside the range of the 923 permutations:", [(M.MORPHS[i], M.MORPHS[j], round(obs[i, j], 3)) for i, j in zip(*np.nonzero(o_rng)) if i < j])
        print("  entries outside 2.5 to 97.5 percentiles:", [(M.MORPHS[i], M.MORPHS[j], round(obs[i, j], 3)) for i, j in zip(*np.nonzero(o_95)) if i < j])
        print(f"  {len(perms)} permutations")
    rare_mean = sum(sums["raw"][m] for m in RARE) / sum(counts[m] for m in RARE)
    freq_mean = sum(sums["raw"][m] for m in FREQ) / sum(counts[m] for m in FREQ)
    print("rare mean raw matrix, row S-tilde = 0:", rare_mean[0].round(3), "; frequent:", freq_mean[0].round(3))
    print("rare mean raw matrix, row S-tilde = 1:", rare_mean[4].round(3), "; frequent:", freq_mean[4].round(3))

**Result: partially reproduced.** The raw rare minus frequent difference has entries outside the
2.5 to 97.5 percentiles of the 923 mouse permutations under both speed readings (S-tilde 0 with
0.25, +0.037; 0.5 with 1, +0.031; 0.75 with 1, +0.034), but no entry of the z-scored difference
(each cell's matrix z-scored, as the Methods say) lies outside the range of all permutations, so
the z-scored part of the criterion fails. Its largest entries, 0 with 0.25 (+0.424) and 0.25 with
0.75 (-0.358, reading a), lie outside the 2.5 to 97.5 percentiles. With the 2020 code's grouping
(R6 counted as frequent) three z-scored entries lie outside the range. In the mean matrices the
rare morph similarity to S-tilde = 0 stays at 0.939 at 0.25 and drops to 0.854 at 0.5, while the
frequent morph similarity drops to 0.902 at 0.25 (reading a), the threshold pattern of the
statement.

## edf6-nmf-confirms-remapping

> Non-negative matrix factorization of single-cell trial-by-trial similarity matrices confirms
> discrete remapping in rare morph sessions and gradual remapping in frequent morph sessions.

In [ ]:
nmf = pd.DataFrame([{"mouse": s["mouse"], "group": s["group"], "day": s["day"], "factors used": len(s["nmf"]["shares"]),
                     "step share": s["nmf"]["step"], "iterations": s["nmf"]["n_iter"]} for s in place])
print(nmf.round(3).to_string(index=False))
print("sessions with no factor whose largest block is (0,0) or (1,1):", int(nmf["step share"].isna().sum()),
      f"; fits stopped at the iteration limit ({M.NMF_MAX_ITER}):", int((nmf.iterations >= M.NMF_MAX_ITER).sum()))
per_mouse = nmf.groupby("mouse")["step share"].mean()
print(per_mouse.round(3).to_dict())
obs, perms = M.mouse_permutation(per_mouse.to_dict(), RARE, FREQ)
print(f"rare mean {per_mouse[RARE].mean():.3f}, frequent mean {per_mouse[FREQ].mean():.3f}; difference {obs:.3f}; "
      f"permutation 2.5 to 97.5 percentiles {np.percentile(perms, 2.5):.3f} to {np.percentile(perms, 97.5):.3f}; "
      f"outside: {bool(M.outside_95(obs, perms))}; range {perms.min():.3f} to {perms.max():.3f}")

**Result: not reproduced.** The step share of the rank-3 NMF factors whose largest block is an
extreme is 0.763 on average over the rare morph mice and 1.072 over the frequent morph mice; the
difference, -0.309, is inside the 2.5 to 97.5 percentiles of the mouse permutations (-0.627 to
0.627) and has the opposite sign to the claim. The frequent mean is raised by F5 (3.399, from a
session at 12.262 whose factor falls little between the extremes); the other mice lie between
0.463 and 1.181. Two sessions have no factor whose largest block is an extreme (F4 session 9, R4
session 10), and one fit stopped at the iteration limit.

## edf5-umap-rare-frequent-separable

> In a nonlinear embedding (UMAP) of the morph-by-position activity of task-modulated cells,
> cells from rare morph and frequent morph mice can be told apart.

In [ ]:
import umap
from sklearn.cluster import DBSCAN, KMeans

t0 = time.time()
X = np.vstack([s["umap_maps"] for s in place])
is_rare = np.concatenate([np.full(s["umap_sig"].sum(), s["group"] == "rare") for s in place])
mouse_of = np.concatenate([np.full(s["umap_sig"].sum(), s["mouse"]) for s in place])
print(f"cells kept by the S-tilde 0 against 1 distance test: {len(X)} of {sum(s['n_cells'] for s in place)} "
      f"(paper 18,230); rare {is_rare.sum()}, frequent {(~is_rare).sum()}")
print("kept per mouse:", pd.Series(mouse_of).value_counts().sort_index().to_dict())
emb = umap.UMAP(n_neighbors=100, metric="correlation", min_dist=0.1, n_components=3, random_state=0).fit_transform(X)
db = DBSCAN(min_samples=200).fit(emb).labels_
print("DBSCAN labels (-1 = between manifolds):", pd.Series(db).value_counts().sort_index().to_dict(), "(paper: 150 dropped)")
manifolds = [k for k in pd.Series(db[db >= 0]).value_counts().index[:2]]
cluster_rows = []
for mi, k in enumerate(manifolds):
    idx = np.where(db == k)[0]
    km = KMeans(10, random_state=0, n_init=10).fit(emb[idx]).labels_
    for c in range(10):
        cells = idx[km == c]
        e10 = umap.UMAP(n_neighbors=20, min_dist=0.1, metric="correlation", n_components=10, random_state=0).fit_transform(X[cells])
        score, perm = M.cv_accuracy_permutations(e10, is_rare[cells], 1000, seed=1000 + 10 * mi + c)
        cluster_rows.append({"manifold": mi, "cluster": c, "cells": len(cells), "rare share": round(is_rare[cells].mean(), 3),
                             "accuracy": round(score, 3), "max of first 100 shuffles": round(perm[:100].max(), 3),
                             "max of 1000 shuffles": round(perm.max(), 3),
                             "sig 100": bool(score > perm[:100].max()), "sig 1000": bool(score > perm.max())})
cr = pd.DataFrame(cluster_rows)
print(cr.to_string(index=False))
print(f"significant clusters: {cr['sig 100'].sum()} of 20 against 100 shuffles, {cr['sig 1000'].sum()} of 20 against 1000")
print(f"section time {time.time() - t0:.0f} s")

In [ ]:
# Variant (2020 code): min_dist 0.01 for the first embedding; only the manifold split is shown.
emb_v = umap.UMAP(n_neighbors=100, metric="correlation", min_dist=0.01, n_components=3, random_state=0).fit_transform(X)
db_v = DBSCAN(min_samples=200).fit(emb_v).labels_
print("variant DBSCAN labels:", pd.Series(db_v).value_counts().sort_index().to_dict())
for k in [k for k in pd.Series(db_v[db_v >= 0]).value_counts().index[:2]]:
    print(f"  manifold {k}: rare share {is_rare[db_v == k].mean():.3f}")
for k in manifolds:
    print(f"main embedding, manifold {k}: rare share {is_rare[db == k].mean():.3f}; overall {is_rare.mean():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, (e, name) in zip(ax, ((emb, "min_dist 0.1"), (emb_v, "min_dist 0.01"))):
    a.scatter(e[~is_rare, 0], e[~is_rare, 1], s=1, c="tab:orange", label="frequent")
    a.scatter(e[is_rare, 0], e[is_rare, 1], s=1, c="tab:blue", alpha=0.5, label="rare")
    a.set_title(name); a.legend(markerscale=8)
plt.show()
print("all points are shown; axes span the embedding:", np.round(emb.min(0), 1).tolist(), np.round(emb.max(0), 1).tolist())

**Result: reproduced.** The distance test keeps 18,742 cells (paper 18,230); DBSCAN splits them
into two manifolds and leaves 274 between them (paper 150). Classification of rare against
frequent cells beats every shuffle in 17 of the 20 clusters against 100 shuffles and in 16 of 20
against 1,000, so readings (a), most clusters, and (b), at least one, hold for both shuffle
counts. Accuracies are 0.519 to 0.708, with rare cells 69% to 87% of a cluster; the classifier
weights classes equally. The 2020 code's min_dist of 0.01 gives the same two-manifold split.

## edf1-licking-before-reward

> Well-trained mice ran steadily down the first half of the track and began licking as they
> approached the reward.

In [ ]:
centers = np.arange(45) * 10 + 5
rbins = [(250, 300), (300, 350), (350, 400)]
for g in "ab":
    print(f"=== speed reading {g} ({'frames below 2 cm/s left out' if g == 'a' else 'all frames'})")
    per_mouse = {}
    first_half = {}
    for s in place:
        lk = s[f"licks_{g}"] * s["rate"]            # licks per frame to licks per second
        rp = s["trials"].reward_pos.to_numpy()
        for (lo, hi) in rbins:
            sel = (rp >= lo) & (rp < hi)
            if sel.any():
                per_mouse.setdefault((s["mouse"], lo), []).append(lk[sel])
        first_half.setdefault(s["mouse"], []).append(lk)
    rows = []
    for (lo, hi) in rbins:
        curves = np.array([np.nanmean(np.vstack(per_mouse[(m, lo)]), axis=0) for m in RARE + FREQ if (m, lo) in per_mouse])
        mean = np.nanmean(curves, axis=0)
        pre = (centers >= lo - 50) & (centers < lo)
        slope = np.polyfit(centers[pre], mean[pre], 1)[0]
        peak = centers[np.nanargmax(mean)]
        rows.append({"reward bin": f"{lo}-{hi}", "mice": len(curves), "rate 50 cm before (first, last bin)":
                     (round(mean[pre][0], 3), round(mean[pre][-1], 3)), "slope per cm": round(slope, 5),
                     "rises": bool(slope > 0 and mean[pre][-1] > mean[pre][0]), "peak at cm": int(peak),
                     "peak within 50 cm": bool(lo - 50 <= peak < hi + 50)})
    print(pd.DataFrame(rows).to_string(index=False))
    fh = {m: np.nanmean(np.vstack(v), axis=0) for m, v in first_half.items()}
    comp = pd.DataFrame({m: {"0-225 cm": np.nanmean(v[centers < 225]), "250-400 cm": np.nanmean(v[(centers >= 250) & (centers < 400)])}
                         for m, v in fh.items()}).T
    comp["lower before"] = comp["0-225 cm"] < comp["250-400 cm"]
    print(comp.round(3).to_string())
    print("mice with lower licking over 0-225 cm:", int(comp["lower before"].sum()), "of", len(comp))
spd = {m: np.nanmean(np.vstack([s["speed_b"] for s in place if s["mouse"] == m]), axis=0) for m in RARE + FREQ}
print("\nMean speed (cm/s) by 50 cm of track, all frames, across mice:",
      np.round([np.mean([spd[m][(centers >= lo) & (centers < lo + 50)].mean() for m in spd]) for lo in range(0, 450, 50)], 1).tolist())
lick_range = np.concatenate([s["licks_b"].ravel() for s in place]) * place[0]["rate"]
print(f"lick rate range over trials and bins: {np.nanmin(lick_range):.2f} to {np.nanmax(lick_range):.2f} licks/s")

**Result: reproduced.** In the across-mouse mean for each 50 cm reward-location bin, lick rate
rises over the 50 cm before the reward location and peaks within 50 cm of it, and every mouse
licks less over 0 to 225 cm than over 250 to 400 cm, under both speed readings (tables above).
The running part of the statement is not judged; mean speed by position is printed.

## Blocked here

edf2-fh-best-predictor. It uses screenshots of the Unity scenes for the range of morph
settings, which are not in the dandiset and not published elsewhere.

supp-earlier-morph-studies. A simulation with priors proposed for other laboratories'
experiments; it uses no data of this paper.

The summary of all claims is at the end of part 2.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min")